In [ ]:
# plot behavior fitting
import os
import matplotlib.pyplot as plt

from general_utils import find_ephys_sessions
from general_visualization import plot_behavior_session
from nwb_utils import NWBUtils
from behavior_utils import get_fitted_model_names

# Get sessions
_, _, sessions = find_ephys_sessions()

error_sessions = []
success_sessions = []

for behavior_session in sessions:
    print(f"Processing session: {behavior_session}")
    try:
        # Read NWB
        nwb_data = NWBUtils.read_behavior_nwb(session_name=behavior_session)

        # Plot behavior
        fig = plot_behavior_session(
            nwb_data=nwb_data,
            model_alias="ForagingCompareThreshold",
            latent_name="right_choice_probability",
        )
        fig = plot_behavior_session(
            nwb_data=nwb_data,
            model_alias=None,
            latent_name=None,
        )

        # Show figure immediately
        plt.show()

        # Optionally close to free memory
        plt.close(fig)

        success_sessions.append(behavior_session)

    except Exception as e:
        # Print error and record the failed session
        print(f"Error in session {behavior_session}: {e}")
        error_sessions.append(behavior_session)
        # Continue to the next session
        continue

print("\nAll sessions:", sessions)
print("Successful sessions:", success_sessions)
print("Error sessions:", error_sessions)

# Save error sessions to a text file
if error_sessions:
    error_log_path = "error_sessions.txt"
    with open(error_log_path, "w") as f:
        for s in error_sessions:
            f.write(f"{s}\n")
    print(f"Error sessions saved to: {error_log_path}")
else:
    print("No error sessions.")


In [ ]:
# behavior qc
# basic behavior
response_rate
opto_session
opto_trials
reward vs no reward fraction 
response timing
win_stay and lose_switch rate
ITI
delay
early lick
volume


linear regression
bias

cut intial and end trials
middle with high no response rate

# different q learning model
model fitting log likelihood each trial
fit goodness conditioned on deltaQ, sumQ, RPE, switch trial
chosenQ, unchosenQ, deltaQ, sumQ, RPE distribution
deltaQ and sumQ correlation
deltaQ and RPE correlation
sumQ and RPE correlation
chosenQ and deltaQ correlation
chosenQ and sumQ correlation
linear repression to predict the choice using the reward and the chosenQ

# compare to threshold model
model fitting log likelihood each trial
fit goodness conditioned on deltaQ, sumQ, RPE, switch trial
value distribution
threshold
linear regression to predict the choice using the reward and the value. 


# define default qc criteria and filter sessions





In [ ]:
nwb_data.trials['animal_response'] 0 left, 1 right, 2 no response
nwb_data.trials['laser_on_trial'] 0 laser off, 1 laser on
nwb_data.trials['rewarded_historyL'] true, false
nwb_data.trials['rewarded_historyR'] true, false
nwb_data.trials['goCue_start_time']
nwb_data.acquisition['left_lick_time']
nwb_data.acquisition['right_lick_time']

nwb_data.trials['reward_size_left']
nwb_data.trials['reward_size_right']

nwb_data.trials['start_time']
nwb_data.trials['delay_start_time']


In [ ]:
from general_utils import find_ephys_sessions
from nwb_utils import NWBUtils
_, _, sessions = find_ephys_sessions()

In [ ]:
nwb_data = NWBUtils.read_behavior_nwb(session_name=sessions[10])

In [ ]:
import importlib
import behavior_qc

importlib.reload(behavior_qc)

from behavior_qc import compute_behavior_qc_from_nwb

metrics, trial_df = compute_behavior_qc_from_nwb(nwb_data)

In [ ]:
import importlib
import behavior_qc_visualization
importlib.reload(behavior_qc_visualization)
from behavior_qc_visualization import (
    plot_behavior_qc_summary,
    plot_qlearning_hist_and_scatter_from_nwb,
    save_combined_behavior_and_qlearning_summary
)


combined_fig, summary = save_combined_behavior_and_qlearning_summary(
    nwb_data,
    save_dir="/root/capsule/scratch/behavior_qc",
    save_basepath="qc_and_qlearning",
    formats=("pdf", "png", "eps"),
)



In [ ]:
import importlib
import behavior_qc_visualization
importlib.reload(behavior_qc_visualization)
from behavior_qc_visualization import (
    plot_behavior_qc_summary,
    plot_qlearning_hist_and_scatter_from_nwb,
)

# Assuming you already have nwb_data loaded:
fig_qc, axes_qc, metrics, trial_df = plot_behavior_qc_summary(nwb_data)

fig_q, axes_q, summary = plot_qlearning_hist_and_scatter_from_nwb(
    nwb_data,
    bins=40,           # optional
)


In [ ]:
import importlib
import behavior_qc_visualization
importlib.reload(behavior_qc_visualization)
from behavior_qc_visualization import (
    plot_behavior_qc_summary,
    plot_qlearning_hist_and_scatter_from_nwb,
    plot_rpe_history_regression_from_nwb
)

fig_hist, axes_hist, coeffs = plot_rpe_history_regression_from_nwb(
    nwb_data,
    max_lag=8,
)


In [ ]:
for col in summary.columns.tolist():
    print(col)


In [1]:
# ============================================
# Batch run behavior QC + Q-learning summary
# for all ephys sessions
# ============================================
import os
import importlib
import matplotlib.pyplot as plt

from general_utils import find_ephys_sessions
from nwb_utils import NWBUtils

import behavior_qc_visualization
importlib.reload(behavior_qc_visualization)
from behavior_qc_visualization import (
    plot_behavior_qc_summary,
    plot_qlearning_hist_and_scatter_from_nwb,
    save_combined_behavior_and_qlearning_summary,
)

# ------------------------------------------------
# Get sessions
# ------------------------------------------------
_, _, sessions = find_ephys_sessions()

error_sessions = []
success_sessions = []

# Base directory for saving outputs
base_save_dir = "/root/capsule/scratch/behavior_qc"

os.makedirs(base_save_dir, exist_ok=True)

for behavior_session in sessions:
    print(f"\nProcessing session: {behavior_session}")
    try:
        # --------------------------
        # Read NWB
        # --------------------------
        nwb_data = NWBUtils.read_behavior_nwb(session_name=behavior_session)

        # --------------------------
        # Run QC + Q-learning summary
        # --------------------------
        # Use session name in save_basepath so files don't overwrite each other
        save_basepath = f"qc_and_qlearning_{behavior_session}"

        combined_fig, summary = save_combined_behavior_and_qlearning_summary(
            nwb_data,
            save_dir=base_save_dir,
            save_basepath=save_basepath,
            formats=("pdf", "png", "eps"),
        )

        # If you do not need to keep the figure open, close to free memory
        plt.close(combined_fig)

        success_sessions.append(behavior_session)

    except Exception as e:
        print(f"Error in session {behavior_session}: {e}")
        error_sessions.append(behavior_session)
        continue

# ------------------------------------------------
# Summary + error log
# ------------------------------------------------
print("\nAll sessions:", sessions)
print("Successful sessions:", success_sessions)
print("Error sessions:", error_sessions)

# Save error sessions to a text file
if error_sessions:
    error_log_path = os.path.join(base_save_dir, "error_sessions_qc_and_qlearning.txt")
    with open(error_log_path, "w") as f:
        for s in error_sessions:
            f.write(f"{s}\n")
    print(f"Error sessions saved to: {error_log_path}")
else:
    print("No error sessions.")



Processing session: ecephys_753124_2024-12-10_17-24-56_sorted_2024-12-13_09-48-25
Found behavior NWB: /root/capsule/data/behavior_nwb/753124_2024-12-10_17-24-56.nwb


/opt/conda/lib/python3.10/site-packages/hdmf/spec/namespace.py:590: UserWarning: Ignoring the following cached namespace(s) because another version is already loaded:
core - cached version: 2.6.0-alpha, loaded version: 2.7.0
The loaded extension(s) may not be compatible with the cached extension(s) in the file. Please check the extension documentation and ignore this warning if these versions are compatible.
  self.warn_for_ignored_namespaces(ignored_namespaces)


Successfully read behavior NWB from: /root/capsule/data/behavior_nwb/753124_2024-12-10_17-24-56.nwb
Plotting alias 'ForagingCompareThreshold' (latent: right_choice_probability)
Can't extract None from None: If fitted_latent is not provided, session_name, model_alias, and latent_name are required
Plotting alias 'None' (latent: None)
Can't extract RPE from None: 'NoneType' object is not subscriptable
Can't extract RPE-1 from None: 'NoneType' object is not subscriptable
Can't extract RPE+1 from None: 'NoneType' object is not subscriptable
Can't extract QL from None: 'NoneType' object is not subscriptable
Can't extract QL-1 from None: 'NoneType' object is not subscriptable
Can't extract QL+1 from None: 'NoneType' object is not subscriptable
Can't extract QR from None: 'NoneType' object is not subscriptable
Can't extract QR-1 from None: 'NoneType' object is not subscriptable
Can't extract QR+1 from None: 'NoneType' object is not subscriptable
Can't extract chosenQ from None: 'NoneType' obje

/opt/conda/lib/python3.10/site-packages/hdmf/spec/namespace.py:590: UserWarning: Ignoring the following cached namespace(s) because another version is already loaded:
core - cached version: 2.6.0-alpha, loaded version: 2.7.0
The loaded extension(s) may not be compatible with the cached extension(s) in the file. Please check the extension documentation and ignore this warning if these versions are compatible.
  self.warn_for_ignored_namespaces(ignored_namespaces)


Successfully read behavior NWB from: /root/capsule/data/behavior_nwb/753125_2024-10-09_10-50-19.nwb
Plotting alias 'ForagingCompareThreshold' (latent: right_choice_probability)
Can't extract None from None: If fitted_latent is not provided, session_name, model_alias, and latent_name are required
Plotting alias 'None' (latent: None)
Can't extract RPE from None: 'NoneType' object is not subscriptable
Can't extract RPE-1 from None: 'NoneType' object is not subscriptable
Can't extract RPE+1 from None: 'NoneType' object is not subscriptable
Can't extract QL from None: 'NoneType' object is not subscriptable
Can't extract QL-1 from None: 'NoneType' object is not subscriptable
Can't extract QL+1 from None: 'NoneType' object is not subscriptable
Can't extract QR from None: 'NoneType' object is not subscriptable
Can't extract QR-1 from None: 'NoneType' object is not subscriptable
Can't extract QR+1 from None: 'NoneType' object is not subscriptable
Can't extract chosenQ from None: 'NoneType' obje

...

Successfully read behavior NWB from: /root/capsule/data/behavior_nwb/753126_2024-10-11_13-14-24.nwb


/root/capsule/src/aind_dft_ephys_analysis/behavior_qc_visualization.py:648: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig = plt.figure(figsize=figsize)


Plotting alias 'ForagingCompareThreshold' (latent: right_choice_probability)
Can't extract None from None: If fitted_latent is not provided, session_name, model_alias, and latent_name are required
Plotting alias 'None' (latent: None)
Can't extract RPE from None: 'NoneType' object is not subscriptable
Can't extract RPE-1 from None: 'NoneType' object is not subscriptable
Can't extract RPE+1 from None: 'NoneType' object is not subscriptable
Can't extract QL from None: 'NoneType' object is not subscriptable
Can't extract QL-1 from None: 'NoneType' object is not subscriptable
Can't extract QL+1 from None: 'NoneType' object is not subscriptable
Can't extract QR from None: 'NoneType' object is not subscriptable
Can't extract QR-1 from None: 'NoneType' object is not subscriptable
Can't extract QR+1 from None: 'NoneType' object is not subscriptable
Can't extract chosenQ from None: 'NoneType' object is not subscriptable
Can't extract chosenQ-1 from None: 'NoneType' object is not subscriptable
Ca

/root/capsule/src/aind_dft_ephys_analysis/behavior_qc_visualization.py:1987: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(
/opt/conda/lib/python3.10/site-packages/hdmf/spec/namespace.py:590: UserWarning: Ignoring the following cached namespace(s) because another version is already loaded:
core - cached version: 2.6.0-alpha, loaded version: 2.7.0
The loaded extension(s) may not be compatible with the cached extension(s) in the file. Please check the extension documentation and ignore this warning if these versions are compatible.
  self.warn_for_ignored_namespaces(ignored_namespaces)


Error in session ecephys_753126_2024-10-11_13-14-24_sorted_2024-11-09_19-18-21: x and y must have same first dimension, but have shapes (11,) and (10,)

Processing session: ecephys_753126_2024-10-15_12-20-35_sorted_2024-11-09_19-47-57
Found behavior NWB: /root/capsule/data/behavior_nwb/753126_2024-10-15_12-20-35.nwb
Successfully read behavior NWB from: /root/capsule/data/behavior_nwb/753126_2024-10-15_12-20-35.nwb
Plotting alias 'ForagingCompareThreshold' (latent: right_choice_probability)
Can't extract None from None: If fitted_latent is not provided, session_name, model_alias, and latent_name are required
Plotting alias 'None' (latent: None)
Can't extract RPE from None: 'NoneType' object is not subscriptable
Can't extract RPE-1 from None: 'NoneType' object is not subscriptable
Can't extract RPE+1 from None: 'NoneType' object is not subscriptable
Can't extract QL from None: 'NoneType' object is not subscriptable
Can't extract QL-1 from None: 'NoneType' object is not subscriptable
Can'